In [2]:
import os
import time
import numpy as np
import pandas as pd
import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm import tqdm

# ============================================================
# CONFIG — matches your real full-run paths/log exactly
# ============================================================
INPUT_DIRS = [
    "/kaggle/input/datasets/anushakirand/rsna-ich-preprocessed-negative-part1",
    "/kaggle/input/datasets/anushakirand/rsna-ich-preprocessed-negative-part2",
    "/kaggle/input/datasets/anushakirand/rsna-ich-preprocessed-negative-part3",
    "/kaggle/input/datasets/anushakirand/rsna-ich-preprocessed-pngs-positive-part-1",
    "/kaggle/input/datasets/anushakirand/rsna-ich-preprocessed-pngs-part3",
    "/kaggle/input/datasets/anushakirand/rsna-ich-preprocessed-pngs",  # positive part 2
]
TRAIN_CSV = "/kaggle/input/competitions/rsna-intracranial-hemorrhage-detection/rsna-intracranial-hemorrhage-detection/stage_2_train.csv"

MIN_FILE_SIZE_KB = 40
MIN_FILE_SIZE_BYTES = MIN_FILE_SIZE_KB * 1024

MODEL_NAME = "convnext_base"

# ---- SMOKE TEST knobs ----
SMOKE_EPOCHS = 3
SMOKE_BATCHES_PER_EPOCH = 50   # cap so this finishes in minutes, not hours
BATCH_SIZE = 64
NUM_WORKERS = 4                # Kaggle T4x2 gives you 4 CPU cores
DEVICE = torch.device("cuda:0")  # single GPU, no DataParallel — that's the point of this test

# ============================================================
# scan_input_dirs — reconstructed from your log's filtering behavior
# (skips PNGs <= MIN_FILE_SIZE_BYTES as low-information)
# ============================================================
def scan_input_dirs(input_dirs, min_size_bytes):
    id_to_path = {}
    skipped_small = 0
    for d in input_dirs:
        print(f"listing {d} ...")
        fnames = [f for f in os.listdir(d) if f.endswith(".png")]
        print(f"  -> {len(fnames)} png files, scanning sizes...")
        for fname in tqdm(fnames, desc=f"scanning {os.path.basename(d)}"):
            full_path = os.path.join(d, fname)
            try:
                size = os.path.getsize(full_path)
            except OSError:
                continue
            if size <= min_size_bytes:
                skipped_small += 1
                continue
            img_id = fname.replace(".png", "")
            id_to_path[img_id] = full_path
    print(f"Scanned {len(input_dirs)} directories: {len(id_to_path)} usable PNGs "
          f"found, {skipped_small} filtered out as low-information (<= "
          f"{min_size_bytes} bytes / {MIN_FILE_SIZE_KB}KB).")
    return id_to_path


# ============================================================
# load_labels — pivots stage_2_train.csv to one row per image,
# columns = subtypes, matching your preprocessing script's pivot logic
# ============================================================
def load_labels(train_csv):
    y = pd.read_csv(train_csv)
    id_split = y.ID.str.rsplit("_", n=1, expand=True)
    y = pd.concat([id_split, y.Label], axis=1)
    y.columns = ["id", "sub_type", "label"]
    y = y.drop_duplicates(subset=["id", "sub_type"])
    df = y.pivot(index="id", columns="sub_type", values="label")
    return df


# ============================================================
# ICHDataset — ASSUMED implementation (I never saw yours — adjust
# if your real class does anything different, e.g. multi-label
# output instead of binary "any", different resize/crop, etc.)
# ============================================================
class ICHDataset(Dataset):
    def __init__(self, df, id_to_path, transform=None):
        self.df = df
        self.id_to_path = id_to_path
        self.transform = transform
        self.ids = list(df.index)

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        path = self.id_to_path[img_id]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        label = float(self.df.loc[img_id, "any"])
        return img, label


# ============================================================
# SMOKE TEST — single GPU, channels_last, proper DataLoader config
# ============================================================
print(f"Smoke test: {SMOKE_EPOCHS} epochs x {SMOKE_BATCHES_PER_EPOCH} batches, "
      f"batch_size={BATCH_SIZE}, workers={NUM_WORKERS}, single GPU (no DataParallel)")

id_to_path = scan_input_dirs(INPUT_DIRS, MIN_FILE_SIZE_BYTES)
if len(id_to_path) == 0:
    raise RuntimeError("No usable PNGs found — check INPUT_DIRS are attached correctly.")

df = load_labels(TRAIN_CSV)
df = df[df.index.isin(id_to_path.keys())]
print(f"Matched {len(df)} images to labels. Positive rate: {df['any'].mean():.4f}")

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                          std=[0.229, 0.224, 0.225]),
])

train_ds = ICHDataset(df, id_to_path, transform=train_transform)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,
    drop_last=True,
)

model = torch.hub.load("pytorch/vision", MODEL_NAME, weights="DEFAULT")
# swap classifier head to 1 logit for binary "any" — adjust if your real
# training targets all 6 subtypes instead of just "any"
if hasattr(model, "classifier"):
    in_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(in_features, 1)

model = model.to(DEVICE, memory_format=torch.channels_last)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4)
scaler = torch.cuda.amp.GradScaler()

model.train()
overall_start = time.time()

for epoch in range(SMOKE_EPOCHS):
    epoch_start = time.time()
    running_loss = 0.0

    for i, (imgs, labels) in enumerate(train_loader):
        if i >= SMOKE_BATCHES_PER_EPOCH:
            break

        iter_start = time.time()

        imgs = imgs.to(DEVICE, non_blocking=True, memory_format=torch.channels_last)
        labels = labels.to(DEVICE, non_blocking=True).float()

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast():
            outputs = model(imgs).squeeze(-1)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()

        if i % 10 == 0:
            torch.cuda.synchronize()  # honest timing only — remove for real training
            iter_time = time.time() - iter_start
            print(f"epoch {epoch+1}/{SMOKE_EPOCHS} batch {i}/{SMOKE_BATCHES_PER_EPOCH} "
                  f"loss={loss.item():.4f} iter_time={iter_time:.2f}s")

    epoch_time = time.time() - epoch_start
    avg_loss = running_loss / min(SMOKE_BATCHES_PER_EPOCH, i + 1)
    print(f"-- epoch {epoch+1} done in {epoch_time:.1f}s, avg_loss={avg_loss:.4f} --")

total_time = time.time() - overall_start
print(f"\nSmoke test complete: {total_time:.1f}s total")
print(f"Projected full epoch time ({len(train_loader)} batches): "
      f"{(total_time / (SMOKE_EPOCHS * SMOKE_BATCHES_PER_EPOCH)) * len(train_loader) / 60:.1f} min/epoch")

Full dataset: 752803 images, 107933 positive (14.3%)
Positive batch: 107933 images
Negative batch: 192067 images

=== Batch 'negative_part3': 64067 files ===
Estimated size at ~40KB/image: ~2.6 GB


negative_part3: 100%|██████████| 64067/64067 [00:00<00:00, 155746.25it/s]


Batch 'negative_part3' — processed: 0, skipped: 64067, failed: 0


In [9]:
import os

count = sum(
    1 for f in os.listdir(OUTPUT_DIR)
    if f.endswith(".png")
)

print("PNG files:", count)

PNG files: 64067


In [13]:
import os

zip_path = "/kaggle/working/negative_part_3.zip"

if os.path.exists(zip_path):
    os.remove(zip_path)
    print("Deleted incomplete ZIP.")
else:
    print("ZIP not found.")


#DO NOT RUN THIS PLEASE!!

ZIP not found.


In [17]:
import os

print(os.path.exists("/kaggle/working/negative_part_3.zip"))

True


In [12]:
import os

total = 0
for root, _, files in os.walk(OUTPUT_DIR):
    for f in files:
        if f.endswith(".png"):
            total += os.path.getsize(os.path.join(root, f))

print(f"Folder size: {total/1e9:.2f} GB")

Folder size: 6.35 GB


In [14]:
zip_output(
    OUTPUT_DIR,
    "/kaggle/working/negative_part_3.zip"
)


Zipping /kaggle/working/png/train_brain_subdural_bone -> /kaggle/working/negative_part_3.zip ...
Zip created: /kaggle/working/negative_part_3.zip (6.36 GB)
Zip verified valid. Deleting source folder /kaggle/working/png/train_brain_subdural_bone to free disk space before the next batch...
Source folder cleared.


In [24]:
import os

print(os.path.exists("/kaggle/working/negative_part_3.zip"))
print(os.listdir("/kaggle/working"))

True
['upload', '.virtual_documents', 'negative_part_2.zip', 'png']


In [27]:
import os

os.makedirs("/kaggle/working/upload", exist_ok=True)

In [28]:
import shutil

shutil.copy(
    "/kaggle/working/negative_part_3.zip",
    "/kaggle/working/upload/negative_part_3.zip"
)

'/kaggle/working/upload/negative_part_2.zip'

In [8]:
%%writefile /kaggle/working/upload/dataset-metadata.json
{
  "title": "RSNA ICH Preprocessed PNGs Negative Part 3",
  "id": "anushakirand/rsna-ich-preprocessed-negative-part3",
  "licenses": [
    {
      "name": "CC0-1.0"
    }
  ]
}

Writing /kaggle/working/upload/dataset-metadata.json


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/upload/dataset-metadata.json'

In [31]:
!kaggle datasets create -p /kaggle/working/upload

Starting upload for file negative_part_2.zip
100%|██████████████████████████████████████| 5.92G/5.92G [04:25<00:00, 24.0MB/s]
Upload successful: negative_part_2.zip (6GB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/anushakirand/rsna-ich-preprocessed-negative-part2


In [34]:
upload_zip = "/kaggle/working/upload/negative_part_3.zip"

if os.path.exists(upload_zip):
    os.remove(upload_zip)
    print("Deleted:", upload_zip)

Deleted: /kaggle/working/upload/negative_part_2.zip
